In [1]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [2]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [3]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [4]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [5]:
import pandas as pd

df_answers = pd.read_csv("data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [6]:
rec = answers[0]
rec

{'question': 'I just found this course late — can I still join now, or is it too late to start?',
 'answer_llm': 'Yes, you can still join now. You can start learning anytime.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [7]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)
print(prompt)

Question:
I just found this course late — can I still join now, or is it too late to start?

Original Answer (ground truth):
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

AI Answer:
Yes, you can still join now. You can start learning anytime.

If you want a certificate, make sure to submit your project while submissions are still open.


In [8]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the key point of the ground truth: late joining is allowed, and certificate eligibility depends on submitting the project before submissions close. The added wording about starting learning anytime is consistent and does not change the meaning.', score='good')

In [9]:
calc_price(usage)

{'input_cost': 0.000228, 'output_cost': 0.0002745, 'total_cost': 0.0005025}

In [10]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [11]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer matches the ground truth: it says the student can still join now and correctly adds the condition that to receive a certificate, the project must be submitted while submissions are still open. This is semantically equivalent.', score='good')

In [12]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [13]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [14]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/515 [00:00<?, ?it/s]

In [15]:
results[10]

({'question': 'How do I join the Office Hours or live workshop stream if I’m a student?',
  'document': '489dd1c9d9',
  'score': 'good',
  'reasoning': 'The AI answer matches the ground truth: students join via YouTube Live, Zoom is only for instructors/presenters/TAs, questions go through Slido, the stream URL is posted in Telegram/Slack announcements before the session, and chat questions may be missed. Extra offer at the end does not change correctness.'},
 ResponseUsage(input_tokens=466, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=79, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=545))

In [16]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [17]:
calc_total_price(usages)

0.35375250000000025

In [18]:
df_eval = pd.DataFrame(evaluations)

In [19]:
df_eval.head()

,question,document,score,reasoning
0,I just found this course late — can I still jo...,74eb249bbf,good,The AI answer preserves the core meaning: late...
1,If I join after the course has already started...,74eb249bbf,good,The AI answer preserves the key point: late jo...
2,What do I need to do to qualify for a certific...,74eb249bbf,good,The AI answer includes the key requirement fro...
3,Is it okay to start the course after it begins...,74eb249bbf,good,The AI answer captures the main idea that star...
4,"When I join late, do I only need to submit the...",74eb249bbf,bad,The AI answer preserves the main point that ce...


In [20]:
df_eval.score.value_counts()

score
good    491
bad      24
Name: count, dtype: int64

In [21]:
df_eval.score.value_counts(normalize=True)

score
good    0.953398
bad     0.046602
Name: proportion, dtype: float64

In [22]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
4,"When I join late, do I only need to submit the...",74eb249bbf,bad,The AI answer preserves the main point that ce...
7,"If I never received an acceptance email, does ...",977bf7786c,bad,The AI answer does not convey the ground truth...
17,How are we supposed to study each week in this...,04919992b3,bad,The AI answer is not semantically equivalent t...
40,When is llm-zoomcamp going to be offered again?,bd31146b0e,bad,The ground truth gives a specific future offer...
42,Is there a planned date for the next llm-zoomc...,bd31146b0e,bad,The ground truth gives a specific planned date...


In [23]:
df_eval.to_csv("data/rag-evaluations-new.csv", index=False)